<a href="https://colab.research.google.com/github/Santosh-S321/Agentic_AI/blob/main/agentic_lab5/LAB_PROGRAM_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Self-Correcting (Adaptive) RAG

In [1]:
!pip install langchain langchain-community langchain-groq chromadb \sentence-transformers pypdf -q

In [2]:
from google.colab import userdata
from langchain_groq import ChatGroq

groq_api_key = userdata.get("GROQ_API_KEY")
llm = ChatGroq(api_key= groq_api_key,
               model="llama-3.1-8b-instant", temperature=0)

### Initialize Vector Store from PDF

Before running the next cell, please:
1.  **Upload your PDF file** to your Colab environment. You can drag and drop it into the files pane on the left, or upload it programmatically.
2.  **Update the `pdf_path` variable** in the code below to the correct path of your uploaded PDF file.

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

# --- IMPORTANT: Upload your PDF to Colab first and then update this path ---
pdf_path = r"/content/IEEE_2024_P2.pdf" # Example: r"/content/your_uploaded_document.pdf"

# Load the PDF document
try:
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
except Exception as e:
    print(f"Error loading PDF: {e}")
    documents = [] # Ensure documents is an empty list if loading fails

if documents:
    # Split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    docs = text_splitter.split_documents(documents)

    # Create embeddings
    embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

    # Initialize Chroma vector store
    db = Chroma.from_documents(docs, embeddings)
    print("Vector store 'db' initialized successfully!")
else:
    db = None
    print("No documents loaded. 'db' variable set to None. Please check your pdf_path and ensure the file exists.")


/tmp/ipykernel_5949/456474659.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/tmp/ipykernel_5949/456474659.py:23: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store 'db' initialized successfully!


In [4]:
def answer_from_pdf(query):
    docs = db.as_retriever().invoke(query)
    context = '\n'.join(d.page_content for d in docs)
    prompt = (f"Use ONLY this context to answer.\n{context}\n\n"
              f"Question: {query}\n"
              "If the context does not contain the answer, reply exactly: "
              "'I don't know'.")
    return llm.invoke(prompt).content

In [5]:
import os

# Get the current working directory
current_directory = os.getcwd()
print(f"Current working directory: {current_directory}")

# List files in the current directory to help locate your PDF
print("Files in current directory:")
for f in os.listdir(current_directory):
    print(f)

Current working directory: /content
Files in current directory:
.config
IEEE_2024_P2.pdf
sample_data


In [6]:
def adaptive_answer(question, max_tries=3):
    query = question
    for attempt in range(1, max_tries + 1):
        print(f'Attempt {attempt} with query: {query}')
        answer = answer_from_pdf(query)
        if "i don't know" not in answer.lower():
            return answer          # good answer, stop
        # otherwise, rephrase and retry
        query = llm.invoke(
            f'Rephrase this search query differently: {query}').content
    return 'Could not find an answer after several tries.'
print(adaptive_answer('What is the main conclusion of the document?'))

Attempt 1 with query: What is the main conclusion of the document?
Attempt 2 with query: Here are a few alternative search queries:

1. What is the key takeaway from the document?
2. What is the author's main point or finding?
3. What is the summary or conclusion of the document?
4. What is the central argument or thesis of the document?
5. What is the final verdict or recommendation of the document?

These rephrased search queries can help you refine your search and get more accurate results.
1. What is the key takeaway from the document?
The key takeaway from the document is that the proposed method (referred to as "Ours1") outperformed the baseline method (referred to as "Baseline1") in terms of Top-1 accuracy.

2. What is the author's main point or finding?
The author's main point is to present a comparison between their proposed method and other studies in the field of SLR (Scene Text Recognition), and to demonstrate the effectiveness of their method.

3. What is the summary or co